# 📊 Issue #1 - Preparación del Dataset

**Asignado a**: Anna (Data & EDA Lead)  
**Issue**: #1 - Preparación del Dataset  
**Fuente**: https://www.football-data.co.uk/Premier/


In [ ]:
# ===============================
# IMPORTACIONES
# ===============================

import pandas as pd
import numpy as np
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración de directorios
DATA_DIR = "data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")


---

# 🔽 PASO 1: Descargar Datos de Premier League


In [ ]:
def descargar_datos_premier_league():
    """
    Descargar datos históricos de Premier League desde 2015 hasta actualidad
    
    Fuente: https://www.football-data.co.uk/Premier/
    
    Returns:
        DataFrame con todos los partidos descargados
    """
    print("\n" + "="*60)
    print("📥 DESCARGANDO DATOS DE PREMIER LEAGUE")
    print("="*60)
    
    dataframes = []
    years_range = list(range(2015, 2024))  # Años disponibles en la fuente
    
    for year in years_range:
        url = f"https://www.football-data.co.uk/Premier/{year}.txt"
        
        try:
            print(f"\n📥 Descargando Premier League {year}...")
            print(f"   URL: {url}")
            
            df = pd.read_csv(url)
            
            dataframes.append(df)
            print(f"   ✅ {len(df)} partidos cargados correctamente")
            
        except Exception as e:
            print(f"   ❌ Error descargando año {year}: {str(e)[:100]}")
    
    # Concatenar todos los DataFrames
    if dataframes:
        df_final = pd.concat(dataframes, ignore_index=True)
        print(f"\n📊 DATASET COMPLETO:")
        print(f"   Total de partidos: {len(df_final)}")
        print(f"   Años cubiertos: {years_range[0]}-{years_range[-1]}")
        
        return df_final
    else:
        print("\n❌ No se pudieron descargar datos.")
        return None

**Nota**: Si la descarga falla por CORS, usaré una alternativa local o simulada con datos históricos.


---

# 🧹 PASO 2: Limpieza de Datos


In [ ]:
def limpiar_datos(df):
    """
    Limpiar y preprocesar los datos
    
    Args:
        df: DataFrame crudo con partidos
        
    Returns:
        DataFrame limpio y listo para análisis
    """
    print("\n" + "="*60)
    print("🧹 LIMPIEZA DE DATOS")
    print("="*60)
    
    # 1. Verificar duplicados
    dups = df.duplicated(subset=['Date', 'HomeTeam', 'AwayTeam']).sum()
    if dups > 0:
        print(f"\n⚠️  Encontrados {dups} duplicados eliminando...")
        df = df.drop_duplicates(subset=['Date', 'HomeTeam', 'AwayTeam'])
    
    # 2. Verificar y corregir valores numéricos (goles)
    print("\n🔢 Corrigiendo columnas de goles...")
    
    # Columna FH (Full Time Home - Goles del local)
    if 'FH' in df.columns:
        df['FH'] = pd.to_numeric(df['FH'], errors='coerce')
        
        # Reemplazar valores no numéricos por -1 o NaN según corresponda
        non_numeric_fh = df['FH'].apply(lambda x: not isinstance(x, (int, float)))
        if non_numeric_fh.any():
            print(f"   ⚠️  Valores no numéricos en FH detectados")
    
    # Columna AH (Full Time Away - Goles del visitante)
    if 'AH' in df.columns:
        df['AH'] = pd.to_numeric(df['AH'], errors='coerce')
        
        non_numeric_ah = df['AH'].apply(lambda x: not isinstance(x, (int, float)))
        if non_numeric_ah.any():
            print(f"   ⚠️  Valores no numéricos en AH detectados")
    
    # 3. Limpieza de columnas de texto
    print("\n📝 Limpieza de columnas de texto...")
    
    for col in ['Date', 'HomeTeam', 'AwayTeam', 'Hteam', 'ATEam']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            # Normalizar mayúsculas/minúsculas para consistencia
            df[col] = df[col].str.lower()
    
    # 4. Eliminar filas con datos inválidos
    print("\n❌ Eliminando registros inválidos...")
    
    if 'FH' in df.columns and 'AH' in df.columns:
        df = df[~(df['FH'].isna() & df['AH'].isna())]  # Eliminar si ambos goles faltan
    
    return df

---

# 💾 PASO 3: Guardar y Verificar Datos


In [ ]:
def info_dataset(df):
    """Mostrar información completa del dataset"""
    print("\n" + "="*60)
    print("📊 INFORMACIÓN DEL DATASET")
    print("="*60)
    
    print(f"\n📦 Tamaño: {len(df)} partidos × {df.shape[1]} columnas")
    print(f"\n📅 Columnas disponibles:")
    for col in df.columns:
        dtype = str(df[col].dtype)
        if df[col].dtype == 'object':
            unique_count = df[col].nunique()
            print(f"   • {col:<15} ({dtype:<8}) - {len(df[col]):>6} valores únicos")
        else:
            print(f"   • {col:<15} ({dtype:<8})")
    
    # Estadísticas por tipo de resultado si existe
    if 'Result' in df.columns:
        print(f"\n📊 Distribución por resultado:")
        for result, count in df['Result'].value_counts().items():
            pct = (count / len(df)) * 100
            print(f"   • {result:<2} ({pct:.1f}%): {count:>6}")

---

# 🎯 EJECUTAR EL SCRIPT


In [ ]:
# Ejecutar el script de preparación del dataset
# Esto descarga, limpia y guarda los datos en data/processed/

print("\n🚀 Iniciando preparación del dataset...")
print("="*60)